<a href="https://colab.research.google.com/github/Mr-KanishakGupta/1BF24CS131_BIS_Lab/blob/main/3_ACO_DroneRouting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random

# 1. Define Environment (10x10 Grid)
GRID_SIZE = 10
START = (0, 0)
GOAL = (9, 9)

# 0 = Free airspace, 1 = Obstacle / No-fly zone
grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
grid[2:5, 2:4] = 1   # Building 1
grid[5:8, 5:7] = 1   # Building 2
grid[1:3, 7:9] = 1   # Restricted Zone

# ACO Parameters
NUM_ANTS = 25
MAX_ITER = 50
ALPHA = 1.0       # Pheromone importance
BETA = 2.5        # Heuristic (distance) importance
RHO = 0.2         # Pheromone evaporation rate
Q = 100.0         # Pheromone deposit factor

# Initialize pheromone matrix for all grid cells
pheromone = np.ones((GRID_SIZE, GRID_SIZE)) * 0.1

# 8-directional movement (dx, dy)
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1),
              (-1, -1), (-1, 1), (1, -1), (1, 1)]

def euclidean_dist(p1, p2):
    return np.hypot(p1[0] - p2[0], p1[1] - p2[1])

def get_valid_neighbors(pos, visited):
    neighbors = []
    for dx, dy in DIRECTIONS:
        nx, ny = pos[0] + dx, pos[1] + dy
        if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
            if grid[nx, ny] == 0 and (nx, ny) not in visited:
                neighbors.append((nx, ny))
    return neighbors

def calculate_path_cost(path):
    cost = 0.0
    for i in range(len(path) - 1):
        cost += euclidean_dist(path[i], path[i+1])
    return cost

# 2. ACO Main Loop
best_path = None
best_cost = float('inf')

for iteration in range(MAX_ITER):
    ant_paths = []

    for ant in range(NUM_ANTS):
        current = START
        path = [START]
        visited = {START}

        while current != GOAL:
            neighbors = get_valid_neighbors(current, visited)
            if not neighbors:
                break  # Dead end

            # Calculate transition probabilities
            probs = []
            for n in neighbors:
                tau = pheromone[n[0], n[1]] ** ALPHA
                # Heuristic: inverse of step distance + distance to goal
                eta = (1.0 / (euclidean_dist(current, n) + euclidean_dist(n, GOAL))) ** BETA
                probs.append(tau * eta)

            probs = np.array(probs)
            probs /= probs.sum()  # Normalize

            # Roulette wheel selection
            next_idx = np.random.choice(len(neighbors), p=probs)
            current = neighbors[next_idx]
            path.append(current)
            visited.add(current)

        # Record valid paths that reached the goal
        if current == GOAL:
            cost = calculate_path_cost(path)
            ant_paths.append((path, cost))
            if cost < best_cost:
                best_cost = cost
                best_path = path

    # 3. Pheromone Update
    pheromone *= (1 - RHO)  # Evaporation
    for path, cost in ant_paths:
        deposit = Q / cost
        for x, y in path:
            pheromone[x, y] += deposit

# 4. Display Results
print(f"Optimal Path Cost (Distance): {best_cost:.2f} units")
print(f"Optimal Waypoints ({len(best_path)} steps):\n{best_path}\n")

# Visualize Grid (. = Free, # = Obstacle, * = Drone Path)
visual = np.full((GRID_SIZE, GRID_SIZE), '.', dtype=str)
visual[grid == 1] = '#'
for x, y in best_path:
    visual[x, y] = '*'
visual[START] = 'S'
visual[GOAL] = 'G'

print("Flight Map:")
for row in visual:
    print(" ".join(row))


Optimal Path Cost (Distance): 14.49 units
Optimal Waypoints (13 steps):
[(0, 0), (0, 1), (1, 2), (1, 3), (1, 4), (2, 5), (3, 5), (4, 6), (5, 7), (6, 8), (7, 8), (8, 8), (9, 9)]

Flight Map:
S * . . . . . . . .
. . * * * . . # # .
. . # # . * . # # .
. . # # . * . . . .
. . # # . . * . . .
. . . . . # # * . .
. . . . . # # . * .
. . . . . # # . * .
. . . . . . . . * .
. . . . . . . . . G
